# Step-by-step implementation
The following are the code implementation of RAG-Fusion. The first five steps are similar to the Multi-Query technique:
  1. Import necessary libraries
  2. Set up the LangSmith and OpenAI API keys
  3. Load and split documents
  4. Index documents
  5. RAG-Fusion: Query generation
  6. Retrieval with reciprocal rank fusion (RRF)
  7. Run the RAG model


## 1. Import necessary libraries

In [1]:
import os
import bs4
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from helpers import get_experientiallabs_llm
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

C:\Users\soura\AppData\Local\Temp\ipykernel_33676\1267605594.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 2. Set up the LangSmith and OpenAI API keys

In [2]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
# os.environ['LANGCHAIN_TRACING_V2'] = 'true'
# os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
# os.environ['LANGCHAIN_API_KEY'] = '' # Add your LangSmith LangChain API key
os.environ['LANGSMITH_PROJECT']='RAG-Fusion'

## 3. Load and split documents

In [4]:
loaders = [
    TextLoader("../../shared_data/blog.langchain.dev_announcing-langsmith_.txt", encoding="utf-8"),
    TextLoader("../../shared_data/blog.langchain.dev_automating-web-research_.txt", encoding="utf-8"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())

In [5]:
docs

[Document(metadata={'source': '../../shared_data/blog.langchain.dev_announcing-langsmith_.txt'}, page_content='URL: https://blog.langchain.dev/announcing-langsmith/\nTitle: Announcing LangSmith, a unified platform for debugging, testing, evaluating, and monitoring your LLM applications\n\nLangChain exists to make it as easy as possible to develop LLM-powered applications.\n\nWe started with an open-source Python package when the main blocker for building LLM-powered applications was getting a simple prototype working. We remember seeing Nat Friedman tweet in late 2022 that there was “not enough tinkering happening.” The LangChain open-source packages are aimed at addressing this and we see lots of tinkering happening now (Nat agrees)–people are building everything from chatbots over internal company documents to an AI dungeon master for a Dungeons and Dragons game.\n\nThe blocker has now changed. While it’s easy to build a prototype of an application in ~5 lines of LangChain code, it’s

In [6]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=400, chunk_overlap=60)
splits = text_splitter.split_documents(docs)

## 4. Index documents

In [7]:
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

## 5. RAG-Fusion: Query generation

In [8]:
template = """You are an AI language model assistant tasked with generating seach queries for a vector search engine.
The user has a question: "{question}"
Your goal/task is to create five variations of this {question} that capture different aspects of the user's intent. These variations will help the search engine retrieve relevant documents even if they don't use the exact keywords as the original question.
Provide these alternative questions, each on a new line.**
Original question: {question}"""

rag_fusion_prompt_template = ChatPromptTemplate.from_template(template)

generate_queries = (
    rag_fusion_prompt_template
    | get_experientiallabs_llm()
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

## 6. Retrieval with reciprocal rank fusion (RRF)

In [9]:
def reciprocal_rank_function(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents
        and an optional parameter k used in the RRF formula """

    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a unique string identifier
            doc_str = str(doc)  # Simple string conversion
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (doc, score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

In [10]:
question = "What is Langsmith, and why do we need it?"
retrieval_chain = generate_queries | retriever.map() | reciprocal_rank_function
docs = retrieval_chain.invoke({"question":question})
len(docs)

6

## 7. Run the RAG model

In [11]:
template = """Answer the following question based on this context:
{context}
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = get_experientiallabs_llm()

final_rag_chain = (
    {"context": retrieval_chain,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

'LangSmith is LangChain’s unified platform for debugging, testing, evaluating, and monitoring LLM-powered applications. It helps developers move applications from unreliable prototypes to production by providing visibility into:\n\n- The inputs and outputs of every model and chain step\n- The sequence of calls and how they are connected\n- Latency, token usage, and costs\n- Errors and unexpected results\n- User interactions and overall application performance\n\nIt also lets teams create datasets from logs, rerun prompts or chains against those datasets, compare changes, and evaluate results using heuristics or other LLMs.\n\nWe need LangSmith because LLM applications are stochastic and difficult to debug or improve reliably. Developers otherwise struggle to understand the exact prompts being sent, what each model call returns, where failures occur, how changes affect outputs, and whether the system is meeting quality, cost, and latency goals. LangSmith brings these workflows together 